# RQ1: Onset from the KNMI 1 km maps, 2016 to 2025, and station maps

This ArcGIS Pro notebook runs the onset calculation on the KNMI High-Resolution Maps for 2016 to 2025 and makes the figure that compares the KNMI stations with the 1 km grid (mean 2017 to 2025, 2017 and 2021).

**Data (not included in this repository):** hourly KNMI High-Resolution Maps per year, the station onsets (`rq1_onset_stations.csv`) and a geodatabase with the station points, the Dutch border and the provinces.

## Step 1: Onset per year from the 1 km maps
Run the onset calculation for every year from 2016 to 2025 and save a raster of the deviation from the national mean per year.

In [ ]:
exec(open('scripts/hrm_onset_multiyear.py', encoding='utf-8').read())

## Step 2: Station points
Compute the mean deviation per KNMI station over 2017 to 2025 and add the stations as points to the project geodatabase.

In [ ]:
import os
import arcpy
import pandas as pd

CSV = os.path.join('data', 'rq1_onset_stations.csv')
OUT_DIR = 'output_rq1_hrm'
gdb = arcpy.mp.ArcGISProject('CURRENT').defaultGeodatabase

d = pd.read_csv(CSV)
years = [f'afw_{y}' for y in range(2017, 2026)]
d['afw_1725'] = d[years].mean(axis=1, skipna=True).round(2)   # missing years are skipped
tmp_csv = os.path.abspath(os.path.join(OUT_DIR, 'stations_dev_1725.csv'))
d[['STN', 'naam', 'lat', 'lon', 'afw_1725', 'afw_2017', 'afw_2021']].to_csv(tmp_csv, index=False)

out = os.path.join(gdb, 'stations_afw')
if arcpy.Exists(out):
    arcpy.management.Delete(out)
arcpy.management.XYTableToPoint(tmp_csv, out, 'lon', 'lat', coordinate_system=arcpy.SpatialReference(4326))
print('Stations:', arcpy.management.GetCount(out)[0])

## Step 3: Map figure
Make the layout with the interpolated station onsets next to the 1 km maps for the mean of 2017 to 2025, for 2017 and for 2021.

In [ ]:
exec(open('scripts/map_stations_vs_1km.py', encoding='utf-8').read())

## Step 4: Pollen station symbol
Give the pollen stations the same name and pin symbol in all maps and export the layout again.

In [ ]:
aprx = arcpy.mp.ArcGISProject('CURRENT')
for m in aprx.listMaps('RQ1_SV_*'):
    for lyr in m.listLayers():
        if lyr.name in ('Pollen trap', 'Pollen station'):
            try:
                cim = lyr.getDefinition('V3')
                cim.renderer.label = 'Pollen station'
                lyr.setDefinition(cim)
            except Exception as e:
                print('Label not set in', m.name, ':', e)
            lyr.name = 'Pollen station'

pin = os.path.abspath(os.path.join(OUT_DIR, 'pollen_pin.lyrx'))
aprx.listMaps('RQ1_SV_2_stations')[0].listLayers('Pollen station')[0].saveACopy(pin)
for m in aprx.listMaps():
    for lyr in m.listLayers():
        if 'poll' in lyr.name.lower():
            arcpy.management.ApplySymbologyFromLayer(lyr, pin)
            lyr.name = 'Pollen station'

aprx.save()
aprx.listLayouts('RQ1_stations_vs_1km')[0].exportToPNG(os.path.abspath(os.path.join(OUT_DIR, 'RQ1_stations_vs_1km.png')), resolution=300)